# Neural Network V7

I compared ReLU and Tanh on the same small train and validation sets.

In [1]:
from pathlib import Path
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch import nn

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_loader import load_task, split_train_val, standardize
from src.neural_net import (
    ModelConfig,
    evaluate,
    parameter_count,
    predict_probabilities,
    train_model,
)

activations = ('relu', 'tanh')
epochs_by_task = {'image': 30, 'sentiment': 5}
device = torch.device('cpu')


In [2]:
def run_activation_sweep(task):
    X, y, _, _ = load_task(task, data_dir=ROOT / 'data')
    X_train, X_val, y_train, y_val = split_train_val(
        X, y, val_fraction=0.2, seed=42
    )
    X_train, y_train = X_train[:1000], y_train[:1000]
    X_val, y_val = X_val[:200], y_val[:200]
    X_train, X_val = standardize(X_train, X_val)

    output_size = 50 if task == 'image' else 3
    rows = []
    predictions_by_activation = {}
    for activation in activations:
        config = ModelConfig(
            hidden_sizes=(64,),
            optimizer='adam',
            learning_rate=0.001,
            batch_size=32,
            activation=activation,
        )
        epochs = epochs_by_task[task]
        started = time.perf_counter()
        result = train_model(
            X_train,
            y_train,
            output_size,
            config,
            epochs,
            seed=42,
            device=device,
            validation_features=X_val,
            validation_labels=y_val,
            early_stopping=False,
        )
        training_time = time.perf_counter() - started

        _, training_accuracy = evaluate(
            result.model,
            X_train,
            y_train,
            config.batch_size,
            device,
            nn.CrossEntropyLoss(),
        )
        prediction_started = time.perf_counter()
        probabilities = predict_probabilities(
            result.model, X_val, config.batch_size, device
        )
        prediction_time = time.perf_counter() - prediction_started
        predictions = probabilities.argmax(axis=1)
        validation_accuracy = float(np.mean(predictions == y_val))

        rows.append({
            'version': 7,
            'task': task,
            'activation': activation,
            'epochs': epochs,
            'best_epoch': result.best_epoch,
            'config': (
                f'hidden_sizes=[64]; activation={activation}; optimizer=adam; '
                f'learning_rate=0.001; batch_size=32; normalized=true; '
                f'training_samples=1000; validation_samples=200'
            ),
            'parameters': parameter_count(result.model),
            'training_time_seconds': training_time,
            'prediction_time_seconds': prediction_time,
            'best_val_accuracy': validation_accuracy,
            'final_train_accuracy': float(training_accuracy),
            'train_val_gap': float(training_accuracy - validation_accuracy),
        })
        predictions_by_activation[activation] = predictions

    best = max(
        rows,
        key=lambda row: (
            row['best_val_accuracy'],
            row['activation'] == 'relu',
        ),
    )
    return {
        'rows': rows,
        'actual_labels': y_val,
        'predictions': predictions_by_activation[best['activation']],
    }


I used the same split and settings as V6, then changed the activation.

In [3]:
image_results = run_activation_sweep('image')
print(pd.DataFrame(image_results['rows']).to_string(index=False))


 version  task activation  epochs  best_epoch                                                                                                                                                 config  parameters  training_time_seconds  prediction_time_seconds  best_val_accuracy  final_train_accuracy  train_val_gap
       7 image       relu      30          26 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       35314              11.662208                 0.009071              0.150                 1.000          0.850
       7 image       tanh      30           6 hidden_sizes=[64]; activation=tanh; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       35314               4.363404                 0.006042              0.135                 0.913          0.778


I started with the image data.

In [4]:
sentiment_results = run_activation_sweep('sentiment')
print(pd.DataFrame(sentiment_results['rows']).to_string(index=False))


 version      task activation  epochs  best_epoch                                                                                                                                                 config  parameters  training_time_seconds  prediction_time_seconds  best_val_accuracy  final_train_accuracy  train_val_gap
       7 sentiment       relu       5           3 hidden_sizes=[64]; activation=relu; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       49411               0.795603                 0.005115              0.610                 0.882          0.272
       7 sentiment       tanh       5           2 hidden_sizes=[64]; activation=tanh; optimizer=adam; learning_rate=0.001; batch_size=32; normalized=true; training_samples=1000; validation_samples=200       49411               1.650803                 0.007970              0.605                 0.793          0.188


Then I did the same test with the sentiment data.

In [5]:
for task, task_results in (
    ('image', image_results),
    ('sentiment', sentiment_results),
):
    summary = pd.DataFrame(task_results['rows'])
    summary.to_csv(
        ROOT / 'results' / task / 'nn_v7_summary.csv',
        index=False,
    )
    pd.DataFrame({
        'validation_index': np.arange(len(task_results['actual_labels'])),
        'actual_label': task_results['actual_labels'],
        'predicted_label': task_results['predictions'],
    }).to_csv(
        ROOT / 'results' / task / 'nn_v7_validation.csv',
        index=False,
    )
    best = max(
        task_results['rows'],
        key=lambda row: (
            row['best_val_accuracy'],
            row['activation'] == 'relu',
        ),
    )
    print(f"{task}: {best['activation']} scored {best['best_val_accuracy']:.3f}")


image: relu scored 0.150
sentiment: relu scored 0.610


I saved both results and the predictions from the better activation.

ReLU did better for both datasets, so I retrained those models with all the labeled examples and saved a submission file.

In [6]:
submission_predictions = []
final_fit_times = {}
for task in ('image', 'sentiment'):
    X, y, X_test, test_ids = load_task(task, data_dir=ROOT / 'data')
    summary = pd.read_csv(ROOT / 'results' / task / 'nn_v7_summary.csv')
    best = summary.loc[summary['best_val_accuracy'].idxmax()]
    X_final, X_test_final = standardize(X, X_test)
    config = ModelConfig(
        hidden_sizes=(64,),
        optimizer='adam',
        learning_rate=0.001,
        batch_size=32,
        activation=best['activation'],
    )
    output_size = 50 if task == 'image' else 3
    started = time.perf_counter()
    result = train_model(
        X_final,
        y,
        output_size,
        config,
        int(best['best_epoch']),
        seed=42,
        device=device,
    )
    final_fit_times[task] = time.perf_counter() - started
    probabilities = predict_probabilities(
        result.model, X_test_final, config.batch_size, device
    )
    labels = probabilities.argmax(axis=1).astype(int)
    if labels.min() < 0 or labels.max() >= output_size:
        raise ValueError(f'{task} predictions are outside the allowed label range')
    submission_predictions.append(pd.DataFrame({
        'id': pd.Series(test_ids, dtype='string'),
        'label': labels,
    }))

all_predictions = pd.concat(submission_predictions, ignore_index=True)
sample = pd.read_csv(ROOT / 'data' / 'sample_submission.csv', dtype={'id': 'string'})
if list(sample.columns) != ['id', 'label']:
    raise ValueError('The sample submission must contain id,label columns')
if len(sample) != 10000 or sample['id'].duplicated().any():
    raise ValueError('The sample submission must have 10000 unique IDs')
if all_predictions['id'].duplicated().any():
    raise ValueError('Test predictions contain duplicate IDs')
if set(sample['id']) != set(all_predictions['id']):
    raise ValueError('Prediction IDs do not match the sample submission IDs')
submission = sample[['id']].merge(
    all_predictions,
    on='id',
    how='left',
    validate='one_to_one',
)
if submission['label'].isna().any() or len(submission) != 10000:
    raise ValueError('Submission is missing one or more test predictions')
submission['label'] = submission['label'].astype(int)
image_rows = submission['id'].str.startswith('img_')
sentiment_rows = submission['id'].str.startswith('text_')
if not image_rows.sum() == 5000 or not sentiment_rows.sum() == 5000:
    raise ValueError('Submission must contain 5000 image and 5000 sentiment IDs')
if not submission.loc[image_rows, 'label'].between(0, 49).all():
    raise ValueError('Image labels must be between 0 and 49')
if not submission.loc[sentiment_rows, 'label'].between(0, 2).all():
    raise ValueError('Sentiment labels must be between 0 and 2')
output_path = ROOT / 'submission' / 'nn' / 'submission_v7_nn.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission)} rows to {output_path}')
print('Full-data refit seconds:', final_fit_times)


Saved 10000 rows to C:\Users\jsidi\OneDrive\Documents\GitHub\COMP472\submission\nn\submission_v7_nn.csv
Full-data refit seconds: {'image': 24.072345399996266, 'sentiment': 1.9084170999994967}
